In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from base_functions import *
from matplotlib.lines import Line2D
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import copy

In [ ]:
def extract_and_organize_data(filepath_ec, filepath_ec_atc, fileday_cpl, fileday_cpl_fe, range_height = True, layer1 = 0, layer2 = 12,
                             minutos = 10, minute1 = None, minute2 = None):
        # Extracting EarthCARE Level 1 and Level 2 data
    
    instance  = Atlid_l2a(filepath_ec)
    ATB355_EC = instance.data.att_bsc_total
    Alt_EC    = instance.data.height/1000.
    Elevation = instance.data.elevation/1000.
    num_layer = instance.data.num_lay
    lay_bot = instance.data.layer_bot
    lay_top = instance.data.layer_top

    instance2  = Atlid_l2a_ATC(filepath_ec_atc)
    Lat_EC    = instance2.data.lat.values
    Lon_EC    = instance2.data.lon.values
    time_EC   = instance2.data.time.values
    Alt_EC    = instance2.data.height/1000.
    Alt_EC_cp = instance2.data.height/1000.
    Elevation = instance2.data.elevation/1000.
    mask_feature = instance2.data.mask
    mask_feature0 = mask_feature.copy(deep = True)
    qc        = instance2.data.qc
    #mask_feature = mask_feature.where(qc <= 1,  np.nan)


    # Extracting CPL Level 1 and Level 2 data
    ds = Dataset(fileday_cpl)

    # This is the CPL Level 1 data
    ATB355_cpl = ds['ATB_355'][:] # Attenuated bz at 355 nm
    ATB532_cpl = ds['ATB_532'][:]  # Attenuated bz at 532 nm
    Mole355_cpl= ds['Mole_Back'][:][0,:,:] #phony_dim_0 = 355nm, phony_dim_1 = 532 nm, phony_dim_3 = 1064 nm 
    Mole532_cpl= ds['Mole_Back'][:][1,:,:] #phony_dim_0 = 355nm, phony_dim_1 = 532 nm, phony_dim_3 = 1064 nm 
    Lat_cpl    = ds['Latitude'][:]
    Lon_cpl    = ds['Longitude'][:]
    Alt_cpl    = ds['Bin_Alt'][:]
    Alt_cpl_cp = ds['Bin_Alt'][:]
    Hour_cpl   = ds['Hour'][:]
    Minute_cpl = ds['Minute'][:]
    Second_cpl = ds['Second'][:]
    Dem_cpl    = ds['DEM_laserspot'][:]/1000.
    day        = fileday_cpl[-20:-18]
    ds.close()

    # Creating a dataframe with the datetime of the CPL Flight
    df = pd.DataFrame({'year': np.transpose([2025]*len(Hour_cpl)),
                       'month': np.transpose([2]*len(Hour_cpl)),
                       'day': np.transpose([day]*len(Hour_cpl)),
                        'hour': Hour_cpl, 
                        'minutes': Minute_cpl, 
                        'seconds': Second_cpl})
    # Changing the dataframe format to be datetime format
    time_cpl = pd.to_datetime(df).values

    # this is the CPL Level 2 data
    ds = Dataset(fileday_cpl_fe)
    feature_matrix = ds['profile']['Feature_Type'][:].T
    cloud_matrix   = ds['profile']['Cloud_Phase'][:].T
    lat_fea        = ds['geolocation']['CPL_Latitude'][:,0]
    lon_fea        = ds['geolocation']['CPL_Longitude'][:,0]

    
    ds.close()

    # Computing the closest approach between EarthCARE and CPL and extracting index where they are closest
    min_idx, closest_air_idx = closest_approach((Lat_EC, Lon_EC), (Lat_cpl, Lon_cpl), sat_time = time_EC, air_time = time_cpl)

    # Let's take 5 minutes around the coincident time and a similar path for EarthCare
    start_time = time_cpl[closest_air_idx] - pd.Timedelta(minutes = minutos)
    end_time   = time_cpl[closest_air_idx] + pd.Timedelta(minutes = minutos)

    # These are the CPL indices
    indices = np.where((time_cpl >= start_time) & (time_cpl <= end_time))[0]

    # These are the EarthCARE indices
    indices_EC = np.where((Lat_EC >= np.min(Lat_cpl[indices])) & (Lat_EC <= np.max(Lat_cpl[indices])))[0]
    mean_height = np.mean(Dem_cpl[indices].data)

    # Now Let's average CPL Data to 1 km resolution, assuming a 200 m horizontal resolution
    # ======================================================================================
    # Selecting the number of profiles of CPL I need to average in order to get the desired horizontal resolution
    nhori = int(np.floor(1./0.2)) # 1 km / 0.2 km

    # Averaging ATB to 1 km using Patrick's function
    ATB355_cpl2 = average_lidar_data(ATB355_cpl, nhori, 0)

    # Averaging the surface elevation
    Elv_cpl2 = np.array([Dem_cpl, Dem_cpl])
    Elv_cpl2 = average_lidar_data(Elv_cpl2.T, nhori, 0)
    
    # Now let's average the time dataset of CPL
    # Convert datetime64 to int for mean computation
    time_as_int = time_cpl.astype('int64')  # nanoseconds since epoch
    
    # Make it 2D to use with your function
    time_as_2d = time_as_int.reshape(1, -1)  # shape (1, N)
    
    # Apply your function to average every 5 values along dimension 1
    mean_time_int = average_lidar_data(time_as_2d, ne=nhori, nd=1)
    
    # Convert back to datetime64
    time_cpl2 = mean_time_int.astype('datetime64[ns]')

    # Computing coincident times between CPL and EarthCARE at 1 km resolution
    if day in ['12']:
        min_idx, closest_air_idx = closest_approach((Lat_EC, Lon_EC), (lat_fea[1:-1], lon_fea[1:-1]), sat_time = time_EC, 
            air_time = time_cpl2[0])
    elif day == '04':
        min_idx, closest_air_idx = closest_approach((Lat_EC, Lon_EC), (lat_fea, lon_fea), sat_time = time_EC, 
            air_time = time_cpl2[0])
    elif day in ['07', '10', '19', '20']:
        min_idx, closest_air_idx = closest_approach((Lat_EC, Lon_EC), (lat_fea[:-1], lon_fea[:-1]), sat_time = time_EC, 
            air_time = time_cpl2[0])

    # Let's take X minutes around the coincident time and a similar path for EarthCare
    if minute1 == None:
        start_time2 = time_cpl2[0][closest_air_idx] - pd.Timedelta(minutes = minutos)
        end_time2   = time_cpl2[0][closest_air_idx] + pd.Timedelta(minutes = minutos)

        # Indices 2 corresponds to the coincident in 10 minutes period track for 1 km resolution 
        indices2 = np.where((time_cpl2[0] >= start_time2) & (time_cpl2[0] <= end_time2))[0]
    else:
        start_time2 = time_cpl2[0][closest_air_idx] + pd.Timedelta(minutes = minute1)
        end_time2   = time_cpl2[0][closest_air_idx] + pd.Timedelta(minutes = minute2)
        if (start_time2 > end_time2):
            indices2 = np.where((time_cpl2[0] >= end_time2) & (time_cpl2[0] <= start_time2))[0]
        else:
            indices2 = np.where((time_cpl2[0] >= start_time2) & (time_cpl2[0] <= end_time2))[0]


    # Computing the difference in time for the CPL track, from the closest approach
    diff_minutes = (time_cpl2[0] - time_cpl2[0][closest_air_idx]) / np.timedelta64(1, 'm')
    
    indices_EC2 = np.where((Lat_EC >= np.min(lat_fea[indices2])) & 
                           (Lat_EC <= np.max(lat_fea[indices2])))[0]
    mean_height2 = np.mean(Dem_cpl[indices2].data)

    # =================================
    #       CPL border selection
    # =================================

    cloud_matrix0 = copy.deepcopy(cloud_matrix)
    if (day == '12') & (layer1 == 3.7) & (layer2 == 7.):
         cloud_matrix[cloud_matrix == 3] = 1.
    if (day == '19') & (layer1 == 3.5) & (layer2 == 7.):
         cloud_matrix[cloud_matrix == 1] = 3.
         cloud_matrix[cloud_matrix == 2] = 3.

    if range_height == True:
        positions_cpl = np.where((Alt_cpl >= layer1) & (Alt_cpl <= layer2))[0]
        cloud_matrix0 = cloud_matrix0[:, positions_cpl]
        cloud_matrix  = cloud_matrix[:, positions_cpl]
        Alt_cpl = Alt_cpl[positions_cpl]
    
    # Now let's compute it but dividing between liquid and ice clouds
    water_cpl = cloud_matrix.T.copy()
    water_cpl[water_cpl != 1] = 0.

    ice_cpl = cloud_matrix.T.copy()
    ice_cpl[ice_cpl != 3] = 0.

    # Taking the borders of ice and water 
    cpl_water_borders = np.diff(water_cpl, axis=0)
    cpl_water_borders = cpl_water_borders.astype(float)
    cpl_water_borders[cpl_water_borders == 0] = np.nan

    cpl_ice_borders = np.diff(ice_cpl, axis=0)
    cpl_ice_borders = cpl_ice_borders.astype(float)
    cpl_ice_borders[cpl_ice_borders == 0] = np.nan

    # Here we compute the cloud top height for water features in CPL
    cloud_top_water_cpl = [] # Define an empty array
    cloud_base_water_cpl = [] # Define an empty array
    for i in np.arange(cpl_water_borders.shape[1]):
        water_cloud_top = Alt_cpl[:][np.where(cpl_water_borders[:,i] == 1)] # 1 because the CPL documentation defines water cloud like this
        water_cloud_bot = Alt_cpl[:][np.where(cpl_water_borders[:,i] == -1)] 
        if len(water_cloud_top) > 0: # If a cloud object is present
            cloud_top_water_cpl.append(np.nanmax(water_cloud_top))
        else:
            cloud_top_water_cpl.append(np.nan)
        if len(water_cloud_bot) > 0: # If a cloud object is present
            cloud_base_water_cpl.append(np.nanmax(water_cloud_bot))
        else:
            cloud_base_water_cpl.append(np.nan)

    # Here we compute the cloud top height for ice features in CPL
    cloud_top_ice_cpl = []  # Define an empty array
    cloud_base_ice_cpl = []
    for i in np.arange(cpl_ice_borders.shape[1]):
        ice_cloud_top = Alt_cpl[:][np.where(cpl_ice_borders[:,i] == 3)]  # 3 because the CPL documentation defines ice cloud like this
        ice_cloud_bot = Alt_cpl[:][np.where(cpl_ice_borders[:,i] == -3)]
        if len(ice_cloud_top) > 0:
            cloud_top_ice_cpl.append(np.nanmax(ice_cloud_top))
        else:
            cloud_top_ice_cpl.append(np.nan)
        if len(ice_cloud_bot) > 0:
            cloud_base_ice_cpl.append(np.nanmax(ice_cloud_bot))
        else:
            cloud_base_ice_cpl.append(np.nan)

    # Creating a dataframe with the cloud base and top for CPL's ice and water clouds 
    df_cpl_border = pd.DataFrame({'ice-top': np.array(cloud_top_ice_cpl).T[indices2],
                              'ice-base': np.array(cloud_base_ice_cpl).T[indices2],
                              'water-top': np.array(cloud_top_water_cpl).T[indices2], 
                              'water-base': np.array(cloud_base_water_cpl).T[indices2]}, index = lat_fea[indices2])
    df_cpl_border = df_cpl_border.sort_index()
    #df_cpl_border[df_cpl_border['water-top'] > 6] = np.nan

    # =================================
    #       EC border selection
    # =================================

    if range_height == True:
        valid_gates = ((Alt_EC >= layer1) & (Alt_EC <= layer2)).all(dim='time')
        # if Alt_EC is 1D (gate only), use directly
        Alt_EC = Alt_EC.isel(gate=valid_gates)
        mask_feature = mask_feature.isel(gate=valid_gates)
        mask_feature0 = mask_feature0.isel(gate=valid_gates)

    opaque = mask_feature.copy(deep = True)
    mask_feature = mask_feature.where(mask_feature != 2, other=1)

    if (day == '07'):
         mask_feature = mask_feature.where(mask_feature != 2, 3)
         mask_feature = mask_feature.where(mask_feature != 1, 3)
    
    # Selecting the water features in EarthCARE. Water is 1 and ice is 2 in simple classification. 
    # But, simple classification 1 is water, 2 is supercooled water and 3 is ice
    cloud_matrix_ec = mask_feature.copy()
    cloud_matrix_ec = cloud_matrix_ec.where(cloud_matrix_ec.isin([1, 2, 3]), 0)

    water_ec = mask_feature.copy()
    water_ec = water_ec.where(water_ec.isin([1]), 0)
    EC_water_borders = water_ec.diff(dim='gate')
    EC_water_borders = EC_water_borders.where(EC_water_borders != 0, np.nan)

    ice_ec = mask_feature.copy()
    ice_ec = ice_ec.where(ice_ec.isin([3]), 0)
    EC_ice_borders = ice_ec.diff(dim='gate')
    EC_ice_borders = EC_ice_borders.where(EC_ice_borders != 0, np.nan)

    # Here we compute the cloud top height for water features in EarthCARE
    cloud_top_water_EC = []
    cloud_base_water_EC = []
    for i in np.arange(EC_water_borders.shape[1]):
        water_cloud_top = Alt_EC[:, i][np.where(EC_water_borders[:,i] == 1)] # water top
        water_cloud_bot = Alt_EC[:, i][np.where(EC_water_borders[:,i] == -1)] # water base
        if len(water_cloud_top) > 0:
            cloud_top_water_EC.append(np.nanmax(water_cloud_top))
        else:
            cloud_top_water_EC.append(np.nan)
        if len(water_cloud_bot) > 0:
            if ((day == '12') & (layer1 == 3.7) & (layer2 == 7)):
                cloud_base_water_EC.append(np.nanmax(water_cloud_bot))
            else:
                cloud_base_water_EC.append(np.nanmin(water_cloud_bot))
        else:
            cloud_base_water_EC.append(np.nan)

    
    # Here we compute the cloud top height for ice features in EarthCARE
    cloud_top_ice_EC = []
    cloud_base_ice_EC = []
    for i in np.arange(EC_ice_borders.shape[1]):
        ice_cloud_top = Alt_EC[:, i][np.where(EC_ice_borders[:,i] == 3)] # water top
        ice_cloud_bot = Alt_EC[:, i][np.where(EC_ice_borders[:,i] == -3)] # water base
        if len(ice_cloud_top) > 0:
            cloud_top_ice_EC.append(np.nanmax(ice_cloud_top))
        else:
            cloud_top_ice_EC.append(np.nan)
        if len(ice_cloud_bot) > 0:
            if ((day == '12') & (layer1 == 3.7) & (layer2 == 7)):
                cloud_base_ice_EC.append(np.nanmax(ice_cloud_bot))
            else:
                cloud_base_ice_EC.append(np.nanmin(ice_cloud_bot))
        else:
            cloud_base_ice_EC.append(np.nan)

    # Creating a dataframe with the cloud base and top for EarthCARE's ice and water clouds 
    df_ec_border = pd.DataFrame({'ice-top': np.array(cloud_top_ice_EC).T[indices_EC2],
                              'ice-base': np.array(cloud_base_ice_EC).T[indices_EC2],
                              'water-top': np.array(cloud_top_water_EC).T[indices_EC2], 
                              'water-base': np.array(cloud_base_water_EC).T[indices_EC2]}, index = Lat_EC[indices_EC2])
    df_ec_border = df_ec_border.sort_index()

    # ========================================================
    #          INTERPOLATING CPL TO EARTHCARE RESOLUTION
    # ========================================================

    # Create an empty DataFrame with df_ec_border's index
    df_interp_cpl_border = pd.DataFrame(index=df_ec_border.index)
    
    # Interpolate each column from df_cpl_border
    for col in df_cpl_border.columns:
        df_interp_cpl_border[col] = np.interp(
            df_ec_border.index,                  # new x (target index)
            df_cpl_border.index,                 # old x
            df_cpl_border[col].values            # old y
        )

    # Interpolating the CPL time difference
    inter_diff_minutes = np.interp(
        df_ec_border.index,
        df_cpl_border.index,
        diff_minutes[indices2]
    )
    
    return (df_ec_border, df_interp_cpl_border, cloud_matrix, indices2, Alt_cpl, mask_feature, 
           indices_EC2, Alt_EC, Lat_EC, inter_diff_minutes, Alt_cpl_cp, Alt_EC_cp, cloud_matrix0, mask_feature0, lat_fea)

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_figure_confusion(A, B, cm, cm0, cm0x, altitude, number, ylim = None, xlim = None):

    fig = plt.figure(figsize=(13.5, 3.8))
    plt.subplots_adjust(wspace=0.25)
    ax2 = fig.add_subplot(131)
    
    for bl in [0,1]: # 0 is baseline BA and 1 is baseline CA
        perfil_cpl   = np.sum(A[bl] == 1, axis = 1)*100/A[bl].shape[1]
        ax2.plot(perfil_cpl, altitude[bl], c='#1E88E5', label='CPL', lw=2.)
        
        perfil_atlid = np.sum(B[bl] == 1, axis = 1)*100/B[bl].shape[1]
        ax2.plot(perfil_atlid, altitude[bl], c='#D81B60' if bl == 0 else '#004D40', 
                 label='EarthCare %s' %('BA' if bl == 0 else 'BD'), 
                 lw=2., ls = '-' if bl == 0 else '--')
        
    custom_lines = [Line2D([0], [0], color='#D81B60', lw=3),
                    Line2D([0], [0], color='#004D40', lw=3, ls = '--'),
                    Line2D([0], [0], color='#1E88E5', lw=3),]
        
    ax2.legend(custom_lines, ['ATLID BA', 'ATLID BD', 'CPL'],
                    fontsize=12.4, loc = 4, ncol = 1)
    
    ax2.tick_params(axis='x', labelsize=14)
    ax2.tick_params(axis='y', labelsize=14)
    ax2.set_xlabel('Cloud area percentage (%)', fontsize= 16)
    ax2.set_xlim(0, 100)
    ax2.set_ylabel('Height (km)', fontsize = 16)
    ax2.set_xticks([10, 30, 50, 70, 90])
    ax2.set_title('a)', fontsize = 16, loc='left')
    if ylim != None:
        ax2.set_ylim(ylim)

    if xlim != None:
        ax2.set_xlim(xlim)

    for bl in [0, 1]:
        ax2 = fig.add_subplot(1,3,2 + bl)
        
        ax2.tick_params(axis='x', labelsize=14)
        ax2.tick_params(axis='y', labelsize=14) 
        
        im = ax2.imshow(cm[bl], cmap = 'Blues', vmin = 0, vmax = 100, alpha = 0.7)
        divider = make_axes_locatable(ax2)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(im, cax = cax)
        # Change tick label font size
        cbar.ax.tick_params(labelsize=12)
        
        classes = ['Clear sky', 'Cloud']
        
        ax2.set_xticks([0,1])
        ax2.set_yticks([0,1])
        
        ax2.set_xticklabels(classes)
        ax2.set_yticklabels(classes, rotation=90, va='center')
        ax2.set_xlabel('ATLID %s' %('BA' if bl == 0 else 'BD'), fontsize = 16)
        if bl == 0:
            ax2.set_ylabel('CPL', fontsize = 16)
        if bl == 1:
            cbar.set_label('Percentage (%)', fontsize = 14)
            ax2.set_yticklabels([])
        
        # Text labels
        for i in range(2):
            for j in range(2):
                ax2.text(
                    j, i,
                    f"{cm0x[bl][i,j]}\n" + f"{cm0[bl][i,j]:.1f}\n" + r"$\bf{" + f"{cm[bl][i,j]:.1f}\%" + r"}$",
                    ha='center',
                    va='center',
                    color='black',
                    fontsize = 14,
                    #weight = 'bold',
                )
        
        ax2.set_title('%s' %('b) baseline BA' if bl == 0 else 'c) baseline BD'), fontsize = 16, loc = 'left')
    plt.savefig('Figure3_paperEC_%s_baselines.png' %number, dpi = 350, bbox_inches = 'tight')

In [ ]:
def select_files(baseline, day):
    if baseline == 'CA':

        path_ebd = '/path_ebd/' # Enter your path here
        filepath04  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250220T214303Z_20250721T124721Z_04169D.h5'
        
        path_atc = '/path_atc/' # Enter your path here
        filepath04_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250204T213904Z_20260909T183408Z_03920D.h5'
        filepath07_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250207T220907Z_20260909T184744Z_03967D.h5'
        filepath10_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250210T223848Z_20260909T184856Z_04014D.h5'
        filepath12_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250212T222744Z_20260909T185011Z_04045D.h5'
        filepath19_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250219T223450Z_20260909T185134Z_04154D.h5'
        filepath20_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250220T214303Z_20260909T185312Z_04169D.h5'
    
    if baseline == 'BA':

        path_ebd = '/path_ebd/' # Enter your path here
        filepath04  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250220T214303Z_20250721T124721Z_04169D.h5'
        
        path_atc = '/path_atc/' # Enter your path here
        filepath04_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250220T214303Z_20250721T124721Z_04169D.h5'

    files_dict = {'4':  [filepath04, filepath04_atc],
                  '7':  [filepath07, filepath07_atc],
                  '10': [filepath10, filepath10_atc],
                  '12': [filepath12, filepath12_atc], 
                  '19': [filepath19, filepath19_atc], 
                  '20': [filepath20, filepath20_atc]}
    
    return files_dict[day]


# This files from GLOVE are in the hard drive!!! 

# Reading CPL processed data
path =  '/path/' # Enter your path here
fileday04 = path + 'ATB_GLOVE_04feb25_merra-2.hdf5'
fileday07 = path + 'ATB_GLOVE_07feb25_merra-2.hdf5'
fileday10 = path + 'ATB_GLOVE_10feb25_merra-2.hdf5'
fileday12 = path + 'ATB_GLOVE_12feb25_merra-2.hdf5'
fileday19 = path + 'ATB_GLOVE_19feb25_merra-2.hdf5'
fileday20 = path + 'ATB_GLOVE_20feb25_merra-2.hdf5'

# Reading CPL feature mask
# New processed Level 2 Layer data
path =  '/path/' # Enter your path here
fileday_fe04 = path + 'CPL_L2_V1-02_01kmPro_259013_04feb25.hdf5'
fileday_fe07 = path + 'CPL_L2_V1-02_01kmPro_259014_07feb25.hdf5'
fileday_fe10 = path + 'CPL_L2_V1-02_01kmPro_259015_10feb25.hdf5'
fileday_fe12 = path + 'CPL_L2_V1-02_01kmPro_259016_12feb25.hdf5'
fileday_fe19 = path + 'CPL_L2_V1-02_01kmPro_259018_19feb25.hdf5'
fileday_fe20 = path + 'CPL_L2_V1-02_01kmPro_259019_20feb25.hdf5'

cpl_dict = {'4':  [fileday04, fileday_fe04],
            '7':  [fileday07, fileday_fe07],
            '10': [fileday10, fileday_fe10],
            '12': [fileday12, fileday_fe12], 
            '19': [fileday19, fileday_fe19], 
            '20': [fileday20, fileday_fe20]}

In [ ]:
for day in ['4']: #, '7']:
    As = []
    Bs = []
    cms = []
    cm0s = []
    cm0xs = []
    altitudes = []
    for baseline in ['BA', 'CA']:

        # Selecting the files
        filepath, filepath_atc = select_files(baseline, day = day)
        fileday, fileday_fe = cpl_dict[day]

        # Retrieving the data and saving 
        date_results = extract_and_organize_data(filepath_ec = filepath,
                                                 filepath_ec_atc = filepath_atc,
                                                 fileday_cpl = fileday, 
                                                 fileday_cpl_fe = fileday_fe, 
                                                 range_height = True, layer1 = 0., layer2 = 10.5, minutos = 10)
        df_ec_border         = date_results[0]
        df_interp_cpl_border = date_results[1]
        cloud_matrix         = date_results[2]
        indices2             = date_results[3]
        Alt_cpl              = date_results[4]
        mask_feature         = date_results[5]
        indices_EC2          = date_results[6]
        Alt_EC               = date_results[7]
        Lat_EC               = date_results[8]
        diff_min             = date_results[9]
        Alt_cpl_copy         = date_results[10]
        Alt_EC_copy          = date_results[11]
        cloud_matrix0        = date_results[12]
        mask_feature0        = date_results[13]
    
        # Creating matrix for CPL with the same spatial grid as ATLID
        cloud_cpl_interp = np.zeros(mask_feature.T[indices_EC2, :].T.shape)
        for i in range(cloud_cpl_interp.shape[1]):
            cth_value_ice   = df_interp_cpl_border['ice-top'].iloc[i]
            cth_value_water = df_interp_cpl_border['water-top'].iloc[i]
            if np.isfinite(cth_value_ice)==True:
                idx1 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['ice-top'].iloc[i]).argmin())
                idx2 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['ice-base'].iloc[i]).argmin())
                cloud_cpl_interp[idx1:idx2, i] = 1
            if np.isfinite(cth_value_water)==True:
                idx1 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['water-top'].iloc[i]).argmin())
                idx2 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['water-base'].iloc[i]).argmin())
                cloud_cpl_interp[idx1:idx2, i] = 1
        cloud_ec = mask_feature0.T[indices_EC2, :].T.copy().data
        cloud_ec[~np.isin(cloud_ec, [1, 2, 3])] = 0
        cloud_ec[np.isin(cloud_ec, [1, 2, 3])] = 1.
        cloud_cpl_interp = cloud_cpl_interp[:,::-1]
    
        B_t = cloud_ec[:,:].copy() 
        A_t = cloud_cpl_interp[:,:].copy() 
        altitude = Alt_EC.T[indices_EC2, :].mean(axis = 0)
        
        TP = np.sum((A_t == 1) & (B_t == 1))
        TN = np.sum((A_t == 0) & (B_t == 0))
        FP = np.sum((A_t == 0) & (B_t == 1))
        FN = np.sum((A_t == 1) & (B_t == 0))
        
        cm = np.array([
            [TN*100/(TN+FP), FP*100/(TN+FP)],
            [FN*100/(FN+TP), TP*100/(FN+TP)]
        ])
        
        cm0 = np.array([
            [TN, FP],
            [FN, TP]
        ])
        
        cm0x = np.array([
            ['TN:', 'FP:'],
            ['FN:', 'TP:']
        ])
        
        CSI = TP / (TP + FP + FN) # Critical Success Index, we want this closer to 1 
        POD = TP / (TP + FN) # Probability of detection
        numerator   = 2 * (TP * TN - FP * FN)
        denominator = (TP + FN) * (FN + TN) + (TP + FP) * (FP + TN)
        HSS = numerator / denominator if denominator > 0 else np.nan

        As.append(A_t)
        Bs.append(B_t)
        cms.append(cm)
        cm0s.append(cm0)
        cm0xs.append(cm0x)
        altitudes.append(altitude)
        
    plot_figure_confusion(As, Bs, cms, cm0s, cm0xs, altitudes, '%s_t' %day, 
                          ylim = (5, 9) if day == '4' else (0,8), 
                          xlim = (0, 50) if day == '4' else (0,95))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.6))
plt.subplots_adjust(wspace=0.25)

for idx, baseline in enumerate(['BA', 'CA']):
    TPs = []
    TNs = []
    FPs = []
    FNs = []
    altitudes = []
    
    POD_total = []
    CSI_total = []

    perfiles_cpl_total = []
    perfiles_atlid_total = []
    for day in ['4', '7', '10', '12', '19', '20']:
        
        # Selecting the files
        filepath, filepath_atc = select_files(baseline, day = day)
        fileday, fileday_fe = cpl_dict[day]

        # Retrieving the data and saving 
        date_results = extract_and_organize_data(filepath_ec = filepath,
                                                 filepath_ec_atc = filepath_atc,
                                                 fileday_cpl = fileday, 
                                                 fileday_cpl_fe = fileday_fe, 
                                                 range_height = True, layer1 = 0., layer2 = 10.5, minutos = 10)
        df_ec_border         = date_results[0]
        df_interp_cpl_border = date_results[1]
        cloud_matrix         = date_results[2]
        indices2             = date_results[3]
        Alt_cpl              = date_results[4]
        mask_feature         = date_results[5]
        indices_EC2          = date_results[6]
        Alt_EC               = date_results[7]
        Lat_EC               = date_results[8]
        diff_min             = date_results[9]
        Alt_cpl_copy         = date_results[10]
        Alt_EC_copy          = date_results[11]
        cloud_matrix0        = date_results[12]
        mask_feature0        = date_results[13]
    
        # Creating matrix for CPL with the same spatial grid as ATLID
        cloud_cpl_interp = np.zeros(mask_feature.T[indices_EC2, :].T.shape)
        for i in range(cloud_cpl_interp.shape[1]):
            cth_value_ice   = df_interp_cpl_border['ice-top'].iloc[i]
            cth_value_water = df_interp_cpl_border['water-top'].iloc[i]
            if np.isfinite(cth_value_ice)==True:
                idx1 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['ice-top'].iloc[i]).argmin())
                idx2 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['ice-base'].iloc[i]).argmin())
                cloud_cpl_interp[idx1:idx2, i] = 1
            if np.isfinite(cth_value_water)==True:
                idx1 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['water-top'].iloc[i]).argmin())
                idx2 = int(np.abs(Alt_EC[:-1].T[indices_EC2, :].T[:, i].values - df_interp_cpl_border['water-base'].iloc[i]).argmin())
                cloud_cpl_interp[idx1:idx2, i] = 1
        cloud_ec = mask_feature0.T[indices_EC2, :].T.copy().data
        cloud_ec[~np.isin(cloud_ec, [1, 2, 3])] = 0
        cloud_ec[np.isin(cloud_ec, [1, 2, 3])] = 1.
        cloud_cpl_interp = cloud_cpl_interp[:,::-1]

        if day == '4':
            # Since February 04 is our first case, We create a matrix to save the result for each day 
            Percentage_matrix_cpl = np.zeros((6,len(Alt_EC_copy[:,0].data)))*np.nan
            Percentage_matrix_ec  = np.zeros((6,len(Alt_EC_copy[:,0].data)))*np.nan
            positions             = np.where(np.isin(Alt_EC_copy[:,0].data,Alt_EC[:,0].data))

        # Now let's count the number of cloudy pixels for both CPL and ATLID
        count_ones_per_altitude_cpl = np.sum(cloud_cpl_interp[:, :] == 1, axis=1)*100/cloud_cpl_interp[:, :].shape[1]
        count_ones_per_altitude_ec  = np.sum(cloud_ec[:, :] == 1, axis=1)*100/cloud_ec[:, :].shape[1]
        
        # Adding the percentages of each day to the total mean
        Percentage_matrix_cpl[0, positions[0]] = count_ones_per_altitude_cpl
        Percentage_matrix_ec[0, positions[0]]  = count_ones_per_altitude_ec

        B_t = cloud_ec[:,:].copy() 
        A_t = cloud_cpl_interp[:,:].copy() 
        altitude = Alt_EC.T[indices_EC2, :].mean(axis = 0)
        
        TP = np.sum((A_t == 1) & (B_t == 1))
        TN = np.sum((A_t == 0) & (B_t == 0))
        FP = np.sum((A_t == 0) & (B_t == 1))
        FN = np.sum((A_t == 1) & (B_t == 0))
        
        cm = np.array([
            [TN*100/(TN+FP), FP*100/(TN+FP)],
            [FN*100/(FN+TP), TP*100/(FN+TP)]
        ])
        
        cm0 = np.array([
            [TN, FP],
            [FN, TP]
        ])
        
        cm0x = np.array([
            ['TN:', 'FP:'],
            ['FN:', 'TP:']
        ])
        
        CSI = TP / (TP + FP + FN) # Critical Success Index, we want this closer to 1 
        POD = TP / (TP + FN) # Probability of detection
        numerator   = 2 * (TP * TN - FP * FN)
        denominator = (TP + FN) * (FN + TN) + (TP + FP) * (FP + TN)
        HSS = numerator / denominator if denominator > 0 else np.nan

        TPs.append(TP)
        TNs.append(TN)
        FPs.append(FP)
        FNs.append(FN)
        altitudes.append(altitude)
        POD_total.append(POD)
        CSI_total.append(CSI)

        # Interpolating to a commom altitude
        new_altitudes = np.arange(0, 10.4, 0.1)
        perfil_cpl   = np.sum(A_t == 1, axis = 1)*100/A_t.shape[1]
        perfil_atlid = np.sum(B_t == 1, axis = 1)*100/B_t.shape[1]
        interp_cpl   = np.interp(new_altitudes, altitude.data[::-1], perfil_cpl[::-1])
        interp_atlid   = np.interp(new_altitudes, altitude.data[::-1], perfil_atlid[::-1])

        perfiles_cpl_total.append(interp_cpl)
        perfiles_atlid_total.append(interp_atlid)

    # Computing average results
    perfil_cpl_total = np.mean(perfiles_cpl_total, axis = 0)
    perfil_atlid_total = np.mean(perfiles_atlid_total, axis = 0)
    
    TP = np.sum(TPs)
    TN = np.sum(TNs)
    FP = np.sum(FPs)
    FN = np.sum(FNs)
    
    cm = np.array([
        [TN*100/(TN+FP), FP*100/(TN+FP)],
        [FN*100/(FN+TP), TP*100/(FN+TP)]
    ])
    
    cm0 = np.array([
        [TN, FP],
        [FN, TP]
    ])
    
    CSI = TP / (TP + FP + FN) # Critical Success Index, we want this closer to 1 
    POD = TP / (TP + FN) # Probability of detection
    numerator   = 2 * (TP * TN - FP * FN)
    denominator = (TP + FN) * (FN + TN) + (TP + FP) * (FP + TN)
    HSS = numerator / denominator if denominator > 0 else np.nan

    # Figure
    axs[idx].tick_params(axis='x', labelsize=14)
    axs[idx].tick_params(axis='y', labelsize=14) 
    axs[0].plot(perfil_cpl_total, new_altitudes, c='#1E88E5', label='CPL', lw=2.)
    axs[0].plot(perfil_atlid_total, new_altitudes, c='#D81B60' if baseline == 'BA' else '#004D40', ls = '-' if baseline == 'BA' else '--', lw=2.)

    im = axs[idx+1].imshow(cm, cmap = 'Blues', vmin = 0, vmax = 100, alpha = 0.7)
    divider = make_axes_locatable(axs[idx+1])
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(im, cax = cax)

    # Change tick label font size
    cbar.ax.tick_params(labelsize=12)

    classes = ['Clear sky', 'Cloud']
    
    axs[idx+1].set_xticks([0,1])
    axs[idx+1].set_yticks([0,1])
    
    axs[idx+1].set_xticklabels(classes)
    axs[idx+1].set_yticklabels(classes, rotation=90, va='center')
    axs[idx+1].set_xlabel('ATLID BA' if baseline == 'BA' else 'ATLID BD', fontsize = 16)

    if baseline == 'BA':
        axs[1].set_ylabel('CPL', fontsize = 16)
    if baseline == 'CA':
        axs[2].tick_params(axis='x', labelsize=14)
        axs[2].tick_params(axis='y', labelsize=14) 
        cbar.set_label('Percentage (%)', fontsize = 14)
        axs[2].set_yticklabels([])
    
    # Text labels
    for i in range(2):
        for j in range(2):
            axs[idx+1].text(j, i, 
                            f"{cm0x[i,j]}\n" + f"{cm0[i,j]:.1f}\n" + r"$\bf{" + f"{cm[i,j]:.1f}\%" + r"}$", 
                            ha='center', 
                            va='center', 
                            color='black', 
                            fontsize = 14,)
    
    axs[idx+1].set_title('b) baseline BA' if baseline == 'BA' else 'c) baseline BD', fontsize = 16, loc = 'left')

custom_lines = [Line2D([0], [0], color='#D81B60', lw=3),
                Line2D([0], [0], color='#004D40', lw=3, ls = '--'),
                Line2D([0], [0], color='#1E88E5', lw=3),]
    
axs[0].legend(custom_lines, ['ATLID BA', 'ATLID BD', 'CPL'],
                fontsize=12.4, loc = 4, ncol = 1)


axs[0].set_xlabel('Cloud area percentage (%)', fontsize= 16)
axs[0].set_ylabel('Height (km)', fontsize = 16)
axs[0].set_xticks([10, 30, 50, 70, 90])
axs[0].set_xlim(0, 50)
axs[0].set_title('a)', fontsize = 16, loc='left')
plt.savefig('Figure10.png', dpi = 350, bbox_inches = 'tight')